In [1]:
import pandas as pd
import warnings
import os
import shutil
warnings.filterwarnings('ignore')
print("✅ Imports done!")

✅ Imports done!


In [2]:
df1 = pd.read_csv("data/A_Z_medicines_dataset_of_india.csv")
df2 = pd.read_csv("data/medDataset_processed.csv")
print(f"✅ Dataset 1: {len(df1)} medicines")
print(f"✅ Dataset 2: {len(df2)} Q&A pairs")

✅ Dataset 1: 253973 medicines
✅ Dataset 2: 16407 Q&A pairs


In [3]:
# Clean dataset 1
df1_clean = df1[['name','price(₹)','manufacturer_name',
                  'type','pack_size_label',
                  'short_composition1','short_composition2']].copy()
df1_clean.fillna("Not available", inplace=True)
df1_clean['text'] = df1_clean.apply(lambda row: 
    f"Medicine: {row['name']}\n"
    f"Price: ₹{row['price(₹)']}\n"
    f"Manufacturer: {row['manufacturer_name']}\n"
    f"Type: {row['type']}\n"
    f"Pack Size: {row['pack_size_label']}\n"
    f"Composition: {row['short_composition1']}, {row['short_composition2']}", 
    axis=1)

# Clean dataset 2
df2_clean = df2[['Question','Answer']].dropna()
df2_clean['text'] = df2_clean.apply(lambda row:
    f"Question: {row['Question']}\nAnswer: {row['Answer']}", axis=1)

print(f"✅ Medicines cleaned: {len(df1_clean)}")
print(f"✅ Q&A cleaned: {len(df2_clean)}")
print(f"Sample medicine:\n{df1_clean['text'][0]}")

✅ Medicines cleaned: 253973
✅ Q&A cleaned: 16407
Sample medicine:
Medicine: Augmentin 625 Duo Tablet
Price: ₹223.42
Manufacturer: Glaxo SmithKline Pharmaceuticals Ltd
Type: allopathy
Pack Size: strip of 10 tablets
Composition: Amoxycillin  (500mg) ,   Clavulanic Acid (125mg)


In [4]:
from langchain_core.documents import Document

all_texts = pd.concat([
    df1_clean[['text']], 
    df2_clean[['text']]
], ignore_index=True)

documents = [
    Document(
        page_content=row['text'],
        metadata={"row_id": idx, "source": "medicine_db"}
    ) 
    for idx, row in all_texts.iterrows()
]

print(f"✅ Total documents: {len(documents)}")
print(f"\nSample doc:\n{documents[0].page_content}")

✅ Total documents: 270380

Sample doc:
Medicine: Augmentin 625 Duo Tablet
Price: ₹223.42
Manufacturer: Glaxo SmithKline Pharmaceuticals Ltd
Type: allopathy
Pack Size: strip of 10 tablets
Composition: Amoxycillin  (500mg) ,   Clavulanic Acid (125mg)


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,   # bigger chunks = more context!
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)
print(f"✅ Total chunks: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0].page_content}")
print(f"\nSample chunk 500:\n{chunks[500].page_content}")
print(f"\nSample chunk 5000:\n{chunks[5000].page_content}")

✅ Total chunks: 323516

Sample chunk:
Medicine: Augmentin 625 Duo Tablet
Price: ₹223.42
Manufacturer: Glaxo SmithKline Pharmaceuticals Ltd
Type: allopathy
Pack Size: strip of 10 tablets
Composition: Amoxycillin  (500mg) ,   Clavulanic Acid (125mg)

Sample chunk 500:
Medicine: Amlodac 2.5 Tablet
Price: ₹61.59
Manufacturer: Zydus Cadila
Type: allopathy
Pack Size: strip of 30 tablets
Composition: Amlodipine (2.5mg), Not available

Sample chunk 5000:
Medicine: Avodox 100mg Capsule
Price: ₹70.0
Manufacturer: Avonic Life Sciences
Type: allopathy
Pack Size: strip of 10 capsules
Composition: Doxycycline (100mg) ,  Lactobacillus (5Billion Spores)


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os

# Verify it's deleted
print(f"Vectorstore exists: {os.path.exists('vectorstore')}")

BATCH_SIZE = 2000
VECTORSTORE_DIR = "vectorstore"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding model loaded!")
print(f"🔧 Building with {len(chunks)} chunks...")

vectorstore = None
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i+BATCH_SIZE]
    if vectorstore is None:
        vectorstore = Chroma.from_documents(
            documents=batch,
            embedding=embeddings,
            persist_directory=VECTORSTORE_DIR
        )
    else:
        vectorstore.add_documents(batch)
    if i % 10000 == 0:
        print(f"✅ {min(i+BATCH_SIZE, len(chunks))}/{len(chunks)}")

print(f"🎉 Done! Total: {vectorstore._collection.count()}")

Vectorstore exists: True


C:\Users\alpha\AppData\Local\Temp\ipykernel_30996\1552778776.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


✅ Embedding model loaded!
🔧 Building with 323516 chunks...
✅ 2000/323516
✅ 12000/323516
✅ 22000/323516
✅ 32000/323516
✅ 42000/323516
✅ 52000/323516
✅ 62000/323516
✅ 72000/323516
✅ 82000/323516
✅ 92000/323516
✅ 102000/323516
✅ 112000/323516
✅ 122000/323516
✅ 132000/323516
✅ 142000/323516
✅ 152000/323516
✅ 162000/323516
✅ 172000/323516
✅ 182000/323516
✅ 192000/323516
✅ 202000/323516
✅ 212000/323516
✅ 222000/323516
✅ 232000/323516
✅ 242000/323516
✅ 252000/323516
✅ 262000/323516
✅ 272000/323516
✅ 282000/323516
✅ 292000/323516
✅ 302000/323516
✅ 312000/323516
✅ 322000/323516
🎉 Done! Total: 684568


In [8]:
# Cell 7 - Test vectorstore
test_queries = [
    "medicine for fever",
    "paracetamol",
    "headache medicine"
]

for query in test_queries:
    results = vectorstore.similarity_search(query, k=3)
    print(f"\n🔍 Query: '{query}'")
    print(f"Found: {len(results)} results")
    for r in results:
        print(f"→ {r.page_content[:120]}")


🔍 Query: 'medicine for fever'
Found: 3 results
→ -  Cancers    -  Autoimmune diseases       Treatment depends on the cause of your fever. Your health care provider may r
→ Question: What is (are) Fever ?
→ Question: What are the treatments for familial Mediterranean fever ?

🔍 Query: 'paracetamol'
Found: 3 results
→ Medicine: Paragreat 250mg Suspension
Price: ₹44.35
Manufacturer: Mankind Pharma Ltd
Type: allopathy
Pack Size: bottle of
→ Medicine: Paragreat 650mg Tablet
Price: ₹30.05
Manufacturer: Mankind Pharma Ltd
Type: allopathy
Pack Size: strip of 15 t
→ Medicine: Fastpara 500mg Tablet
Price: ₹10.5
Manufacturer: Santiago Lifesciences Pvt Ltd
Type: allopathy
Pack Size: stri

🔍 Query: 'headache medicine'
Found: 3 results
→ Question: What are the treatments for Headache ?
→ Question: What is the outlook for Headache ?
→ Question: What are the treatments for Migraine ?


In [9]:
from langchain_community.llms import Ollama

llm = Ollama(model="mistral", temperature=0.3)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

def ask(question):
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])
    prompt = f"""You are MediBot India, a helpful medical assistant specialized in Indian medicines.
Use the context below to answer the question.
List specific medicine names, prices and compositions when relevant.
If you cannot find the answer, say so clearly.
Always remind users to consult a real doctor.

Context:
{context}

Question: {question}

Answer:"""
    return llm.invoke(prompt)

print("✅ Q&A Chain ready!")

# Quick test
print("\n🧪 Testing...")
result = ask("medicine for fever")
print(f"Answer: {result}")

C:\Users\alpha\AppData\Local\Temp\ipykernel_30996\477072812.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="mistral", temperature=0.3)


✅ Q&A Chain ready!

🧪 Testing...
Answer:  In the context provided, here are the treatments mentioned for the specified conditions:

1. Fever (not a specific disease): Over-the-counter medicines such as acetaminophen, ibuprofen, and aspirin (for adults) can be used to lower a high fever. It's important to consult a doctor before taking any medication.

2. Familial Mediterranean Fever: The treatment typically involves medication when they have a fever to prevent another febrile seizure. Over-the-counter anti-inflammatory medicines like aspirin and ibuprofen may be prescribed.

3. Rheumatic Fever: Treatment for rheumatic fever includes rest, over-the-counter anti-inflammatory medicines such as aspirin and ibuprofen, and antibiotics like penicillin to treat the underlying streptococcal infection. The treatment duration is usually 7 to 14 days, but symptoms like headache, weakness, and malaise may persist for weeks after adequate treatment.

4. Medicine for fever in children: Over-the-count

In [18]:
import gradio as gr
from gradio import ChatMessage

def ask_assistant(question, history):
    if not question.strip():
        return history, ""
    response = ask(question)
    history = history + [
        ChatMessage(role="user", content=question),
        ChatMessage(role="assistant", content=response)
    ]
    return history, ""

def clear_chat():
    return [], ""

css = """
body { background: linear-gradient(135deg, #0f0c29, #302b63, #24243e) !important; }
footer {display: none !important}
"""

with gr.Blocks(css=css, title="💊 MediBot India") as demo:
    gr.HTML("""
    <div style="text-align:center; padding:30px 0 10px 0;">
        <div style="font-size:3em;">💊</div>
        <div style="font-size:2.2em; font-weight:900;
                    background:linear-gradient(90deg, #00c6ff, #0072ff);
                    -webkit-background-clip:text;
                    -webkit-text-fill-color:transparent;">
            MediBot India
        </div>
        <div style="color:#a0aec0; margin-top:5px;">
            Powered by Mistral 7B • 100% Offline
        </div>
        <div style="margin-top:10px; padding:8px 20px;
                    background:rgba(255,100,100,0.15);
                    border:1px solid #ff6464; border-radius:20px;
                    display:inline-block; color:#ff6464; font-size:0.9em;">
            ⚠️ Educational purposes only — Always consult a real doctor!
        </div>
    </div>
    """)
    gr.HTML("""
    <div style="display:flex; justify-content:center; gap:15px; margin:20px 0;">
        <div style="background:rgba(0,198,255,0.1); border:1px solid #00c6ff;
                    border-radius:12px; padding:12px 20px; text-align:center;">
            <div style="font-size:1.5em; font-weight:bold; color:#00c6ff;">323K+</div>
            <div style="color:#a0aec0; font-size:0.85em;">Medical Documents</div>
        </div>
        <div style="background:rgba(0,198,255,0.1); border:1px solid #00c6ff;
                    border-radius:12px; padding:12px 20px; text-align:center;">
            <div style="font-size:1.5em; font-weight:bold; color:#00c6ff;">253K+</div>
            <div style="color:#a0aec0; font-size:0.85em;">Indian Medicines</div>
        </div>
        <div style="background:rgba(0,198,255,0.1); border:1px solid #00c6ff;
                    border-radius:12px; padding:12px 20px; text-align:center;">
            <div style="font-size:1.5em; font-weight:bold; color:#00c6ff;">16K+</div>
            <div style="color:#a0aec0; font-size:0.85em;">Medical Q&As</div>
        </div>
        <div style="background:rgba(118,75,162,0.2); border:1px solid #764ba2;
                    border-radius:12px; padding:12px 20px; text-align:center;">
            <div style="font-size:1.5em; font-weight:bold; color:#b794f4;">100%</div>
            <div style="color:#a0aec0; font-size:0.85em;">Offline & Private</div>
        </div>
    </div>
    """)
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                height=420,
                label="💬 Chat with MediBot"
            )
            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="💊 Ask about any medicine...",
                    label="", scale=5, container=False
                )
                submit_btn = gr.Button("Ask 🚀", variant="primary", scale=1)
            clear_btn = gr.Button("🗑️ Clear Chat", size="sm")
        with gr.Column(scale=1):
            gr.Markdown("### ⚡ Quick Questions")
            q1 = gr.Button("💊 Paracetamol uses?", size="sm")
            q2 = gr.Button("💰 Price of Augmentin 625?", size="sm")
            q3 = gr.Button("🤒 Medicines for fever?", size="sm")
            q4 = gr.Button("⚠️ Azithromycin side effects?", size="sm")
            q5 = gr.Button("🔬 Medicines with Amoxicillin?", size="sm")
            q6 = gr.Button("😴 Medicines for sleep?", size="sm")
            gr.Markdown("---")
            gr.Markdown("""
### 🔧 Tech Stack
🤖 **LLM:** Mistral 7B
🔍 **Search:** MiniLM
🗄️ **DB:** ChromaDB
🦜 **Framework:** LangChain
🖥️ **GPU:** RTX 4050
            """)

    submit_btn.click(ask_assistant, inputs=[question_box, chatbot], outputs=[chatbot, question_box])
    question_box.submit(ask_assistant, inputs=[question_box, chatbot], outputs=[chatbot, question_box])
    clear_btn.click(clear_chat, outputs=[chatbot, question_box])
    q1.click(lambda h: ask_assistant("What is Paracetamol used for?", h), inputs=[chatbot], outputs=[chatbot, question_box])
    q2.click(lambda h: ask_assistant("Price of Augmentin 625?", h), inputs=[chatbot], outputs=[chatbot, question_box])
    q3.click(lambda h: ask_assistant("What medicines are used for fever?", h), inputs=[chatbot], outputs=[chatbot, question_box])
    q4.click(lambda h: ask_assistant("Side effects of Azithromycin?", h), inputs=[chatbot], outputs=[chatbot, question_box])
    q5.click(lambda h: ask_assistant("What medicines contain Amoxicillin?", h), inputs=[chatbot], outputs=[chatbot, question_box])
    q6.click(lambda h: ask_assistant("What medicines help with sleep?", h), inputs=[chatbot], outputs=[chatbot, question_box])

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
